# Grants
- `ingenieros_datos`: lectura y escritura en bronze, silver y golden
- `analistas`: solo lectura en golden

Si un grupo no existe, el grant se omite y se registra en el log.

In [0]:
%run ./00_utils

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("grupo_ingenieros", "ingenieros_datos")
dbutils.widgets.text("grupo_analistas", "analistas")

CATALOGO = dbutils.widgets.get("catalogo")
ING = dbutils.widgets.get("grupo_ingenieros")
ANA = dbutils.widgets.get("grupo_analistas")

In [0]:
GRANTS = [
    f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `{ING}`",
    *[f"GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA {CATALOGO}.{capa} TO `{ING}`" for capa in CAPAS],
    f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `{ANA}`",
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOGO}.golden TO `{ANA}`",
]

omitidos = []
for sentencia in GRANTS:
    try:
        spark.sql(sentencia)
        log(sentencia, entidad="grants")
    except Exception as e:
        if "PRINCIPAL_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
            log(f"Grupo no existe, se omite: {sentencia}", "WARN", "grants")
            omitidos.append(sentencia)
        else:
            raise

if omitidos:
    log(f"{len(omitidos)} grants omitidos", "WARN", "grants")

In [0]:
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.golden"))